# CredAIR — Análise de Risco de Crédito

**Objetivo:** usar o histórico de propostas de crédito da CredAIR para entender os fatores ligados à inadimplência e construir um modelo de **Regressão Logística** que identifique propostas de maior risco.

## 0. Configuração do ambiente

Importamos as bibliotecas e fixamos uma **semente aleatória** (`RANDOM_STATE`). Com ela, a divisão treino/teste e o treinamento produzem sempre os mesmos resultados.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Semente aleatória: garante resultados reprodutíveis
RANDOM_STATE = 42

# Caminhos das camadas (o notebook roda dentro de notebooks/, então a raiz fica um nível acima)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRONZE = RAIZ / "data" / "bronze" / "dataset.csv"
PRATA = RAIZ / "data" / "silver" / "credito_limpo.parquet"
OURO = RAIZ / "data" / "gold" / "credito_modelagem.parquet"
FIGURAS = RAIZ / "reports" / "figures"

# Estilo dos gráficos: fundo limpo e duas cores fixas para as classes
AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#898781"
CORES_CLASSE = {0: AZUL, 1: LARANJA}
sns.set_theme(style="whitegrid", rc={"grid.color": "#e1e0d9", "axes.edgecolor": "#c3c2b7"})
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)


def salvar(fig, nome):
    # Salva a figura em reports/figures para usar na apresentação
    fig.savefig(FIGURAS / f"{nome}.png", dpi=120, bbox_inches="tight")


print("Bronze encontrada:", BRONZE.exists())


Bronze encontrada: True


## 1. Entendimento do problema

### 1.1 Carregando os dados brutos (camada bronze)

Lemos o arquivo **exatamente como foi recebido**. Nenhuma alteração é feita nesta etapa.


In [2]:
bronze = pd.read_csv(BRONZE)
print(f"Linhas: {bronze.shape[0]:,} | Colunas: {bronze.shape[1]}")
bronze.head()


Linhas: 5,000 | Colunas: 11


,idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
0,52,MS,Ensino Superior,"2,911.820",14,690,"5,934.190",60,Viagem,1,0
1,49,RR,Ensino Médio,"3,176.430",5,676,"6,791.380",24,Outros,2,0
2,48,AL,Ensino Superior,"3,429.400",23,558,"14,666.780",24,Outros,5,1
3,59,GO,Ensino Fundamental,"10,096.700",2,739,"19,053.240",24,Reforma,0,0
4,37,RJ,Ensino Fundamental,"3,033.250",24,797,"6,171.310",6,Educação,0,1


### 1.2 Variável resposta e tipo de problema

- **Variável resposta:** `inadimplente`. Vale `1` quando a proposta resultou em falta de pagamento e `0` quando não.
- **Tipo de problema:** **classificação binária supervisionada**. É *supervisionada* porque temos o resultado histórico de cada proposta, e *binária* porque a resposta tem duas classes.


In [3]:
contagem = bronze["inadimplente"].value_counts().sort_index()
pd.DataFrame({
    "quantidade": contagem,
    "proporção": (contagem / contagem.sum()).map("{:.1%}".format),
}).rename(index={0: "0 - adimplente", 1: "1 - inadimplente"})


,quantidade,proporção
inadimplente,,
0 - adimplente,3280,65.6%
1 - inadimplente,1720,34.4%


**Primeira leitura:** cerca de **34%** das propostas resultaram em inadimplência. As classes são **moderadamente desbalanceadas**. Um "modelo" que dissesse sempre "adimplente" acertaria cerca de 66% dos casos **sem identificar nenhum inadimplente**. Por isso a **acurácia não é uma boa métrica** aqui: usaremos ROC-AUC e recall.


## 2. Diagnóstico de qualidade dos dados

Antes de corrigir qualquer coisa, fazemos um **inventário dos problemas**.

> 🧭 **Decisão: diagnosticar antes de tratar.** Separar as duas etapas evita "consertar" um problema e esconder outro.

### 2.1 Tipos e valores nulos


In [4]:
pd.DataFrame({
    "tipo": bronze.dtypes.astype(str),
    "nulos": bronze.isna().sum(),
    "valores_distintos": bronze.nunique(),
})


,tipo,nulos,valores_distintos
idade,int64,0,73
estado,str,0,85
escolaridade,str,0,4
renda_mensal,float64,0,4921
tempo_emprego_anos,int64,0,32
score_credito,int64,0,428
valor_solicitado,float64,0,4937
prazo_meses,int64,0,7
finalidade,str,0,6
dividas_ativas,int64,0,8


**Leitura da tabela:**
- **Não há valores nulos explícitos.** Isso não garante que os dados estejam corretos: um problema pode aparecer como **valor inválido** em vez de vazio (verificado na seção 2.2).
- ⚠️ **`estado` tem 85 valores distintos, mas o Brasil tem só 27 UFs.** Hipótese: variações de escrita da mesma UF, como `"MG"`, `" MG"` e `"MG "`. Para o computador, são categorias diferentes. Confirmado na seção 2.4.
- `escolaridade` (4) e `finalidade` (6) têm poucas categorias, como esperado. `prazo_meses` tem só 7 valores, então funciona quase como uma categoria.
- `renda_mensal` e `valor_solicitado` têm quase um valor distinto por linha, o esperado para valores com centavos. Por isso, duas linhas idênticas até nos centavos dificilmente são coincidência (verificado na seção 2.3).
- As colunas de texto (`estado`, `escolaridade` e `finalidade`) serão convertidas para **categoria** na limpeza.


### 2.2 Estatísticas descritivas


In [6]:
bronze.describe().T


,count,mean,std,min,25%,50%,75%,max
idade,"5,000.000",47.767,31.646,18.000,32.000,46.000,61.000,750.000
renda_mensal,"5,000.000","5,228.634","3,968.478",100.000,"2,697.320","4,146.300","6,458.877","44,814.190"
tempo_emprego_anos,"5,000.000",12.662,7.518,1.000,6.000,13.000,19.000,74.000
score_credito,"5,000.000",652.481,84.817,335.000,594.000,653.000,710.000,850.000
valor_solicitado,"5,000.000","14,264.657","8,268.261","1,721.480","8,467.170","12,365.630","17,764.885","50,000.000"
prazo_meses,"5,000.000",29.922,15.483,6.000,18.000,24.000,36.000,60.000
dividas_ativas,"5,000.000",1.900,1.865,0.000,0.000,1.000,3.000,7.000
inadimplente,"5,000.000",0.344,0.475,0.000,0.000,0.000,1.000,1.000


Alguns sinais aparecem já na tabela:
- `idade` chega a **750 anos**.
- `tempo_emprego_anos` chega a **74 anos**.
- `renda_mensal` tem mínimo de **R$ 100**.

Não há nulos explícitos, mas um problema pode aparecer como **valor inválido** em vez de vazio.


### 2.3 Duplicidades


In [7]:
n_dup = bronze.duplicated().sum()
print(f"Linhas duplicadas (idênticas em todas as colunas): {n_dup}")
bronze[bronze.duplicated(keep=False)].sort_values(list(bronze.columns)).head(6)


Linhas duplicadas (idênticas em todas as colunas): 19


,idade,estado,escolaridade,renda_mensal,tempo_emprego_anos,score_credito,valor_solicitado,prazo_meses,finalidade,dividas_ativas,inadimplente
813,20,RN,Ensino Superior,"7,393.680",13,826,"20,006.350",36,Reforma,2,0
3723,20,RN,Ensino Superior,"7,393.680",13,826,"20,006.350",36,Reforma,2,0
2748,23,SC,Ensino Médio,"5,657.970",3,718,"6,468.510",36,Reforma,1,0
4726,23,SC,Ensino Médio,"5,657.970",3,718,"6,468.510",36,Reforma,1,0
2177,25,AC,Ensino Médio,"4,663.760",10,715,"12,020.380",36,Veículo,0,0
3089,25,AC,Ensino Médio,"4,663.760",10,715,"12,020.380",36,Veículo,0,0


**Leitura:** há **19 linhas idênticas em todas as 11 colunas**, inclusive renda e valor com centavos. Pela tabela 2.1, esses valores quase nunca se repetem, então a chance de duas propostas diferentes coincidirem em tudo é praticamente nula. Isso indica **registro duplicado**: as cópias estão espalhadas pela base (ex.: linhas 813 e 3723), o que é compatível com uma proposta que entrou duas vezes no sistema.

**Por que tratar:** uma duplicata pesa em dobro no treino e, se uma cópia cair no treino e outra no teste, o modelo é avaliado num exemplo que já viu (**vazamento de dados**).
**Tratamento proposto (seção 3):** remover as cópias, mantendo a primeira ocorrência.


### 2.4 Categorias com espaços extras


In [8]:
for col in ["estado", "escolaridade", "finalidade"]:
    valores = bronze[col]
    com_espaco = (valores != valores.str.strip()).sum()
    print(f"{col:13s} | categorias distintas: {valores.nunique():3d} | "
          f"após remover espaços: {valores.str.strip().nunique():3d} | registros com espaços: {com_espaco}")

print("\nExemplos:", bronze.loc[bronze["estado"] != bronze["estado"].str.strip(), "estado"].map(repr).unique()[:6])


estado        | categorias distintas:  85 | após remover espaços:  27 | registros com espaços: 101
escolaridade  | categorias distintas:   4 | após remover espaços:   4 | registros com espaços: 0
finalidade    | categorias distintas:   6 | após remover espaços:   6 | registros com espaços: 0

Exemplos: <ArrowStringArray>
['' DF '', '' PA '', '' BA '', '' PI '', ''  CE '', '' AM  '']
Length: 6, dtype: str


**Leitura:** ✅ hipótese da seção 2.1 confirmada. `estado` cai de **85 para 27 categorias** ao remover os espaços: os 58 "estados" a mais eram variações como `' DF '` e `'  CE '`, em **101 registros**. `escolaridade` e `finalidade` não têm o problema.

**Tratamento proposto (seção 3):** remover espaços e padronizar em maiúsculas. Isso deve ser feito **antes** de remover as duplicatas, porque duas linhas que só diferem no espaço do estado também são cópias.


### 2.5 Valores inválidos e inconsistências

Cada regra de negócio vira um teste. Contamos quantos registros violam cada uma.


In [10]:
idade_valida = bronze["idade"].between(18, 100)

regras = {
    "idade fora de 18–100 anos": ~idade_valida,
    "tempo de emprego ≥ idade": bronze["tempo_emprego_anos"] >= bronze["idade"],
    "começou a trabalhar antes dos 14 anos (idade válida)":
        idade_valida & (bronze["tempo_emprego_anos"] > bronze["idade"] - 14),
    "renda mensal < R$ 600": bronze["renda_mensal"] < 600,
    "score fora de 0–1000": ~bronze["score_credito"].between(0, 1000),
    "valor solicitado ≤ 0": bronze["valor_solicitado"] <= 0,
    "dívidas ativas < 0": bronze["dividas_ativas"] < 0,
}
pd.DataFrame({"registros": {k: int(v.sum()) for k, v in regras.items()}})


,registros
idade fora de 18–100 anos,15
tempo de emprego ≥ idade,8
começou a trabalhar antes dos 14 anos (idade válida),696
renda mensal < R$ 600,10
score fora de 0–1000,0
valor solicitado ≤ 0,0
dívidas ativas < 0,0


**Leitura:**
- **Impossíveis (erro certo):** 15 idades fora de 18–100 anos e 8 casos de tempo de emprego ≥ idade.
- **Suspeitos:** 10 rendas abaixo de R$ 600.
- **Plausíveis, mas incomuns:** 696 casos (14% da base) de início de trabalho antes dos 14 anos. O volume é alto demais para tratar sem investigar.
- ✅ Score, valor solicitado e dívidas ativas passaram em todas as verificações.


In [11]:
print("Idades fora do intervalo:", sorted(bronze.loc[~idade_valida, "idade"]))
print("Rendas abaixo de R$ 600:  ", sorted(bronze.loc[bronze["renda_mensal"] < 600, "renda_mensal"]))

Idades fora do intervalo: [210, 280, 300, 340, 360, 430, 440, 500, 540, 660, 670, 680, 700, 710, 750]
Rendas abaixo de R$ 600:   [100.0, 150.0, 200.0, 250.0, 300.0, 350.0, 400.0, 450.0, 500.0, 550.0]


**Leitura:**
- **Idades:** todas as 15 inválidas são **múltiplas de 10** (210 a 750). Dividir por 10 daria idades plausíveis (21 a 75), sugerindo um zero digitado a mais. Testamos essa correção: 14 casos ficam coerentes, mas 1 passa a indicar início de trabalho aos 11 anos. Como a correção é uma **suposição não verificável** e o volume é pequeno (0,3%), optamos por **marcar como ausente** e imputar no pipeline.
- **Rendas:** as 10 rendas abaixo de R$ 600 formam uma **sequência exata** de R$ 100 a R$ 550, de 50 em 50, um registro de cada. Isso não ocorre naturalmente: são **valores artificiais**. Tratamento: **marcar como ausente**.


#### Investigação: início de trabalho antes dos 14 anos


In [12]:
# Onde estão os casos de "começou a trabalhar antes dos 14 anos"?
inconsistente = regras["começou a trabalhar antes dos 14 anos (idade válida)"]
faixa = pd.cut(bronze["idade"], [17, 25, 35, 45, 100], labels=["18–25", "26–35", "36–45", "46+"])
pd.DataFrame({
    "casos": inconsistente.groupby(faixa, observed=True).sum(),
    "% da faixa": inconsistente.groupby(faixa, observed=True).mean().map("{:.0%}".format),
})


,casos,% da faixa
idade,,
18–25,351,50%
26–35,314,36%
36–45,25,3%
46+,6,0%


**Leitura:** o problema está **concentrado nos jovens**: atinge 50% das pessoas de 18 a 25 anos e 36% das de 26 a 35, mas é raro acima dos 36.

> 🧭 **Decisão: manter esses casos (P6).** São cerca de 690 registros (14% da base). Removê-los ou anulá-los **enviesaria a base contra os jovens**, e o modelo funcionaria pior justamente para esse grupo. Há explicações plausíveis (aprendiz, trabalho informal ou familiar), então o caso é **incomum, não impossível**. Fica registrado como **limitação** da análise.


### 2.6 Resumo do diagnóstico

| # | Problema | Registros | Evidência | Tratamento proposto (seção 3) |
|---|---|---|---|---|
| P1 | Espaços extras em `estado` | 101 | 85 categorias → 27 após remover espaços | Remover espaços e padronizar em maiúsculas |
| P2 | Linhas duplicadas | 19 | Idênticas nas 11 colunas, inclusive centavos | Remover cópias (depois do P1) |
| P3 | Idade fora de 18–100 anos | 15 | 210 a 750, todas múltiplas de 10 | Marcar como ausente |
| P4 | Tempo de emprego ≥ idade | 8 | Impossível | Marcar como ausente |
| P5 | Renda abaixo de R$ 600 | 10 | Sequência exata de R$ 100 a R$ 550 | Marcar como ausente |
| P6 | Início de trabalho antes dos 14 anos | 696 | Concentrado em 18–35 anos | **Manter** e registrar como limitação |

**Por que "marcar como ausente" em vez de remover a linha?** As outras 10 colunas da linha estão corretas. Apagar a linha inteira jogaria fora informação boa por causa de um único campo errado.
